In [1]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from PreProcessorML import PreProcessorML
from ModelCollection import ModelCollection
from PipelineML import PipelineML
from CrossValidation import CrossValidation

In [2]:
## Loading data
data_loader = DataLoader(path_train="../data/labeledTrainData.csv", path_test="../data/testData.csv")
data_splitter = DataSplitter(target_column="sentiment")
train, test = data_loader.load()
train.shape, test.shape

((25000, 3), (25000, 2))

In [3]:
train

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...
...,...,...,...
24995,3453_3,0,It seems like more consideration has gone into...
24996,5064_1,0,I don't believe they made this film. Completel...
24997,10905_3,0,"Guy is a loser. Can't get girls, needs to buil..."
24998,10194_3,0,This 30 minute documentary Buñuel made in the ...


In [4]:
## splitting data
X, y = train.drop(columns=['sentiment']), train['sentiment']
X_train, X_test, y_train, y_test = data_splitter.split(train)
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_train.shape[0] / (X_test.shape[0] + X_train.shape[0])

((20000, 2), (5000, 2), (20000,), (5000,), 0.8)

In [5]:
## ML pipeline builder
pipeline_builder = PipelineML(train)

### Logistic Regression

In [6]:
params = {'l1_ratio': 0, 'solver': 'saga', 'C': 10.0, 'max_iter': 1000}
pipeline = pipeline_builder.build(model_name="Logistic Regression", params=params)
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('nlp', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the

In [7]:
## Hold-Out Validation score (one fold only)
cv = CrossValidation(pipeline=clone(pipeline))

cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
print("Holdout Validation score: ", cv_score)

## K-fold Cross-Validation 
cv_score = cv.evaluate(X, y)
print("K-fold CV score: ", cv_score)

Fit time: 45.00470209121704
Prediction time: 4.932844877243042

Holdout Validation score:  0.905
K-fold CV score:  {'mean_rmse': np.float64(0.90504), 'std_rmse': np.float64(0.004068218283229162)}


In [10]:
## Hyper-parameter tunning
param_grid = {
    "model__C": [1e-4, 1e-2, 1, 10],
    "model__l1_ratio": [0],
}

cv_lg_results = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                             reduced=True, train_size=5000, return_loss_curves=False)
cv_lg_results

Params:  {'model__C': 0.0001, 'model__l1_ratio': 0}
Fit time: 15.014885663986206
Prediction time: 5.759066581726074

Params:  {'model__C': 0.01, 'model__l1_ratio': 0}
Fit time: 9.236612558364868
Prediction time: 4.923105478286743

Params:  {'model__C': 1, 'model__l1_ratio': 0}
Fit time: 8.662252426147461
Prediction time: 5.100586414337158

Params:  {'model__C': 10, 'model__l1_ratio': 0}
Fit time: 11.359908103942871
Prediction time: 4.772071838378906

Total CV time: 64.84625697135925



,model__C,model__l1_ratio,fit_time,pred_time,epochs,final_loss,final_validation_score,score
3,10.0000,0,11.359908,4.772072,NaN,NaN,NaN,0.8840
2,1.0000,0,8.662252,5.100586,NaN,NaN,NaN,0.8660
1,0.0100,0,9.236613,4.923105,NaN,NaN,NaN,0.7914
0,0.0001,0,15.014886,5.759067,NaN,NaN,NaN,0.4962


In [11]:
## Create submission using best hyper-parameters
preds = cv.pipeline.fit(X, y).predict(test)
submission = pd.DataFrame({"id": test['id'], "sentiment": preds})
submission.to_csv("../data/submission_lg.csv", index=False)